In [1]:
from src.preprocessing.document_parser import DocumentParser
from src.ingestion.graph_ingestion import GraphDBManager
from src.ingestion.vector_ingestion import VectorDBManager

from langchain_ollama import OllamaLLM
import src.utils.constants as C

c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_paths = [
	"data/raw/WickedRose_andNCPH.pdf",
]

In [3]:
parser = DocumentParser({})

In [4]:
# parsed_content = parser.parse_batch(file_paths)
parsed_content = parser.load_parsed_content(["data/processed/WickedRose_andNCPH_parsed.pkl"])

Loaded: data/processed/WickedRose_andNCPH_parsed.pkl


In [5]:
print(f"\nParsed {len(parsed_content)} documents successfully")
for pc in parsed_content:
	print(f"  - {pc.source_file}: "
			f"{len(pc.text.get('content', ''))} chars text, "
			f"{len(pc.images)} images, "
			f"{len(pc.tables)} tables")


Parsed 1 documents successfully
  - data/raw/WickedRose_andNCPH.pdf: 17497 chars text, 13 images, 0 tables


In [6]:
print("\n" + "=" * 80)
print("STEP 2: Initializing LLM")
print("=" * 80)

llm = OllamaLLM(
	model="mistral",
	temperature=0.0,
	base_url="http://localhost:11434"
)
print("LLM initialized: Ollama (mistral)")



STEP 2: Initializing LLM
LLM initialized: Ollama (mistral)


In [7]:
graph_manager = GraphDBManager(
    llm=llm,
    uri=C.NEO4J_URI,
    username=C.NEO4J_USERNAME,
    password=C.NEO4J_PASSWORD,
    database=C.NEO4J_DB,
    allowed_nodes=[],
    allowed_relationships=[]
)

2025-11-12 11:35:53,449 - GraphDBManager - INFO - Connected to Neo4j at neo4j://127.0.0.1:7687
2025-11-12 11:35:53,453 - GraphDBManager - INFO - LLM Graph Transformer initialized
2025-11-12 11:35:53,583 - GraphDBManager - INFO - Constraint created: entity_id
2025-11-12 11:35:53,645 - GraphDBManager - INFO - Constraint created: document_id
2025-11-12 11:35:53,708 - GraphDBManager - INFO - Constraint created: chunk_id
2025-11-12 11:35:53,773 - GraphDBManager - INFO - Constraint created: image_id
2025-11-12 11:35:53,833 - GraphDBManager - INFO - Constraint created: table_id


In [8]:
graph_stats = graph_manager.ingest_batch_parsed_content(parsed_content)
print(f"\nGraph Ingestion Complete:")
print(f"  - Documents: {graph_stats['documents']}")
print(f"  - Text Chunks: {graph_stats['text_chunks']}")
print(f"  - Images: {graph_stats['images']}")
print(f"  - Tables: {graph_stats['tables']}")
print(f"  - Entities: {graph_stats['entities']}")
print(f"  - Relationships: {graph_stats['relationships']}")

2025-11-12 11:36:12,960 - GraphDBManager - WARNING - LLM extraction failed for chunk 2: unhashable type: 'list'
Traceback (most recent call last):
  File "c:\Users\ishani.kathuria\projects\TrustworthyRAG\src\ingestion\graph_ingestion.py", line 201, in _ingest_text_chunks
    graph_docs = self.graph_transformer.convert_to_graph_documents([doc])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\langchain_experimental\graph_transformers\llm.py", line 932, in convert_to_graph_documents
    return [self.process_response(document, config) for document in documents]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\langchain_experimental\graph_transformers\llm.py", line 863, in process_response
    nodes_set.add((rel["tail"], rel.get("tail_type", DEFAULT_NODE_TYPE)))
TypeError: unhashable type: 'list'
2025-11-


Graph Ingestion Complete:
  - Documents: 1
  - Text Chunks: 28
  - Images: 13
  - Tables: 0
  - Entities: 280
  - Relationships: 185


In [9]:
vector_manager = VectorDBManager(
    uri=C.NEO4J_URI,
    username=C.NEO4J_USERNAME,
    password=C.NEO4J_PASSWORD,
    database=C.NEO4J_DB,
    text_embedding_model="all-MiniLM-L6-v2"
)

2025-11-12 11:38:01,005 - VectorDBManager - INFO - Connected to Neo4j at neo4j://127.0.0.1:7687
c:\Users\ishani.kathuria\projects\TrustworthyRAG\src\ingestion\vector_ingestion.py:38: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.text_embedder = HuggingFaceEmbeddings(
c:\Users\ishani.kathuria\projects\TrustworthyRAG\src\ingestion\vector_ingestion.py:50: LangChainDeprecationWarning: The class `Neo4jVector` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jVector``.
  se

In [10]:
embedding_stats = vector_manager.batch_embed_parsed_contents(parsed_content)
print(f"\nEmbedding Complete:")
print(f"  - Text Embeddings: {embedding_stats['text_embeddings']}")
print(f"  - Image Embeddings: {embedding_stats['image_embeddings']}")
print(f"  - Table Embeddings: {embedding_stats['table_embeddings']}")
print(f"  - Entity Embeddings: {embedding_stats['entity_embeddings']}")

2025-11-12 11:38:09,799 - VectorDBManager - INFO - Embedding complete for data/raw/WickedRose_andNCPH.pdf: {'text_embeddings': 28, 'image_embeddings': 13, 'table_embeddings': 0, 'entity_embeddings': 279}
2025-11-12 11:38:09,800 - VectorDBManager - INFO - Batch embedding complete: {'text_embeddings': 28, 'image_embeddings': 13, 'table_embeddings': 0, 'entity_embeddings': 279}



Embedding Complete:
  - Text Embeddings: 28
  - Image Embeddings: 13
  - Table Embeddings: 0
  - Entity Embeddings: 279


In [11]:
test_query = "What is GinWui?"
print(f"\nQuery: '{test_query}'")

results = vector_manager.similarity_search_multimodal(
    query=test_query,
    modality="all",
    k=3
)

print(f"\nFound {len(results)} results:")
for idx, result in enumerate(results):
    print(f"\n  Result {idx + 1} ({result['modality']}):")
    print(f"    Content: {result['content'][:200]}...")
    print(f"    Metadata: {result['metadata']}")


Query: 'What is GinWui?'

Found 9 results:

  Result 1 (text):
    Content: ## 3 The GinWui Backdoor Rootkit Payload...
    Metadata: {'chunk_index': 6, 'modality': 'text'}

  Result 2 (text):
    Content: Zero day attacks commenced in May 2006 attempted to install a GinWui backdoor Trojan horse and Windows rootkit. A DLL file called��wuis.dll and several SYS files install themselves when a computer is ...
    Metadata: {'chunk_index': 7, 'modality': 'text'}

  Result 3 (text):
    Content: remote control" rootkit code on May 2, 2006. This distribution of GinWui was largely unknown and undetected by anti-virus companies at the time of release.
Versions of GinWui used in targeted attacks ...
    Metadata: {'chunk_index': 8, 'modality': 'text'}

  Result 4 (image):
    Content: NCHP 5.0 Screenshot (GinWui Rootkit)...
    Metadata: {'path': 'data\\extracted_images\\WickedRose_andNCPH_page2_0_NCHP 50 Screenshot GinWui Rootkit.png', 'modality': 'image', 'bbox': '[0.208, 0.584, 0.791, 0.906

In [14]:
test_query = "Who created GinWui?"
print(f"\nQuery: '{test_query}'")

results = vector_manager.similarity_search_multimodal(
    query=test_query,
    modality="all",
    k=3
)

print(f"\nFound {len(results)} results:")
for idx, result in enumerate(results):
    print(f"\n  Result {idx + 1} ({result['modality']}):")
    print(f"    Content: {result['content'][:200]}...")
    print(f"    Metadata: {result['metadata']}")


Query: 'Who created GinWui?'

Found 9 results:

  Result 1 (text):
    Content: ## 3 The GinWui Backdoor Rootkit Payload...
    Metadata: {'chunk_index': 6, 'modality': 'text'}

  Result 2 (text):
    Content: remote control" rootkit code on May 2, 2006. This distribution of GinWui was largely unknown and undetected by anti-virus companies at the time of release.
Versions of GinWui used in targeted attacks ...
    Metadata: {'chunk_index': 8, 'modality': 'text'}

  Result 3 (text):
    Content: ## "Wicked Rose"

From an ancient Chinese poem, expressing the devotion of his heart for hacking.
Charles: "Silence belongs to our world..."
"After you choose the technology you love, you have to rese...
    Metadata: {'chunk_index': 21, 'modality': 'text'}

  Result 4 (image):
    Content: NCHP 5.0 Screenshot (GinWui Rootkit)...
    Metadata: {'path': 'data\\extracted_images\\WickedRose_andNCPH_page2_0_NCHP 50 Screenshot GinWui Rootkit.png', 'modality': 'image', 'bbox': '[0.208, 0.584, 0.791, 

In [12]:
final_stats = graph_manager.get_statistics()
print(f"\nGraph Database Statistics:")
print(f"  - Total Entities: {final_stats.get('total_entities', 0)}")
print(f"  - Total Relations: {final_stats.get('total_relations', 0)}")


Graph Database Statistics:
  - Total Entities: 227
  - Total Relations: 503


In [13]:
# CALL db.index.vector.queryNodes("entity_vector_index", 3, [-0.1438547968864441, 0.03441644087433815, -0.048849254846572876, -0.057281963527202606, -0.021827775985002518, -0.06383774429559708, 0.018905404955148697, 0.02431485243141651, 0.0013411047402769327, 0.00978201162070036, 0.007012942340224981, 0.03507915511727333, 0.03168350085616112, 0.004671454895287752, -0.055443521589040756, -0.015419846400618553, -0.07622459530830383, 0.016178591176867485, -0.01061366405338049, -0.006955572869628668, 0.02951197884976864, -0.009354900568723679, -0.025129690766334534, -0.021553266793489456, 0.03575776144862175, -0.015435836277902126, -0.018736084923148155, 0.0318380743265152, 0.030695877969264984, 0.017640244215726852, 0.017020925879478455, 0.04208294302225113, 0.09146927297115326, -0.013922669924795628, -0.01072425115853548, 0.03739442303776741, 0.028528163209557533, -0.05171414837241173, 0.016492458060383797, 0.04890400171279907, 0.0019144117832183838, 0.018612610176205635, -0.03148766979575157, 0.08221650123596191, -0.03749287500977516, -0.01012451108545065, -0.04220319911837578, 0.0358380526304245, 0.06129888445138931, -0.01975264772772789, -0.049034059047698975, 0.025208638980984688, -0.002060773316770792, -0.03266361355781555, -0.016540005803108215, -0.11182384938001633, 0.0017089447937905788, -0.018364280462265015, 0.08687940239906311, 0.09440542757511139, 0.05605287477374077, 0.049657199531793594, -0.03508635237812996, 0.08853133022785187, 0.000327045563608408, -0.008679892867803574, -0.03499109670519829, 0.05001627281308174, 0.0016386422794312239, -0.028488822281360626, 0.040145132690668106, -0.0068212696351110935, -0.012590309605002403, 0.06326711922883987, 0.04422644153237343, 0.006920505780726671, -0.03632492944598198, 0.016780011355876923, -0.02559669502079487, -0.08021854609251022, 0.026939352974295616, 0.007640357594937086, 0.08865675330162048, 0.06149338185787201, 0.01152827125042677, 0.07438624650239944, 0.04065756872296333, 0.010166271589696407, 0.1074880063533783, 0.04673295095562935, -0.004298352636396885, 0.0006101717590354383, 0.05844864621758461, 0.028834722936153412, 0.06450662761926651, 0.019206929951906204, 0.05349935591220856, -0.08914078027009964, -0.04017007350921631, 0.10552499443292618, -0.08753255754709244, -0.052648212760686874, 0.07769739627838135, -0.08361369371414185, 0.027866190299391747, 0.03870859369635582, 0.040635786950588226, 0.031856101006269455, 0.09812494367361069, -0.00031914495048113167, -0.01662250980734825, -0.000903660838957876, -0.018279768526554108, -0.08199042826890945, 0.002586684888228774, 0.013297537341713905, -0.013540076091885567, 0.06395012885332108, -0.039858877658843994, 0.009212306700646877, 0.04290303587913513, -0.019378650933504105, -0.016497930511832237, -0.09466604143381119, -0.067286916077137, -0.07260603457689285, 0.0023960440885275602, -3.300989807858546e-33, -0.031010020524263382, -0.06897495687007904, -0.07466808706521988, -0.01737915351986885, -0.003236608812585473, -0.044861841946840286, -0.023540379479527473, -0.039718396961688995, -0.06436754018068314, -0.0049056424759328365, -0.03315579146146774, -0.0033329459838569164, -0.031670551747083664, 0.06909056007862091, 0.0812370702624321, 0.031293705105781555, -0.000496574561111629, 0.05711947754025459, -0.03715880587697029, -0.014310403726994991, 0.10574248433113098, 0.012507339008152485, -0.0333884060382843, -0.034670937806367874, -0.018087519332766533, 0.04015378654003143, 0.02941071055829525, -0.05314279720187187, 0.05941738188266754, 0.043592557311058044, -0.053794268518686295, -0.03183227777481079, -0.05221936106681824, -0.07552650570869446, -0.006332510616630316, 0.012314616702497005, -0.05871519073843956, -0.07100837677717209, -0.00766321225091815, -0.051213350147008896, 0.052984192967414856, -0.020625947043299675, -0.035171207040548325, -0.03127451613545418, 0.00857439823448658, -0.028892168775200844, -0.07361456751823425, -0.005909394472837448, -0.021606281399726868, 0.06745821982622147, -0.05117165297269821, 0.048489056527614594, 0.006891652010381222, 0.08031303435564041, -0.11329542100429535, 0.033334970474243164, 0.04140467196702957, 0.01549869030714035, 0.036980245262384415, 0.01723012886941433, 0.025425054132938385, 0.0025517463218420744, -0.01616377755999565, -0.03192746266722679, -0.036345627158880234, -0.036062899976968765, 0.05004125460982323, 0.005657381843775511, -0.017112722620368004, 0.06443128734827042, -0.09152427315711975, -0.01752506010234356, -0.07713036239147186, 0.0242020171135664, -0.11595852673053741, -0.07035818696022034, 0.0941409021615982, -0.002768979175016284, -0.055846989154815674, -0.02575903758406639, -0.0570950023829937, -0.033555321395397186, 0.03695570304989815, -0.002141882898285985, -0.06411650031805038, 0.01800421066582203, -0.006351314950734377, -0.03091225028038025, -0.07732900232076645, 0.08906448632478714, 0.013801860623061657, -0.014155428856611252, -0.0016237797681242228, 0.06142343208193779, -0.06631124764680862, 1.1354379114500795e-33, -0.06254663318395615, -0.03727162629365921, 0.03599175438284874, -0.02182619459927082, -0.13244327902793884, 0.02142786793410778, 0.018581753596663475, 0.06392862647771835, -0.09423980861902237, 0.03573966771364212, 0.013724449090659618, 0.06245921924710274, 0.08037152886390686, -0.007824091240763664, 0.04232971742749214, 0.11436660587787628, 0.08263453841209412, 0.007437453139573336, -0.11285654455423355, 0.01817578263580799, 0.017637142911553383, 0.10386772453784943, -0.05890008062124252, -0.03371761739253998, -0.06513730436563492, 0.01700281724333763, 0.015442072413861752, 0.07788705080747604, -0.026461904868483543, 0.07421422749757767, 0.11727685481309891, 0.04037465900182724, -0.1550520360469818, 0.022889364510774612, 0.04581774026155472, -0.027759643271565437, 0.10862820595502853, -0.04153229668736458, -0.020409051328897476, -0.08703707158565521, 0.09569433331489563, 0.05106164887547493, -0.0007943868986330926, 0.055030085146427155, 0.028242740780115128, 0.03304452449083328, -0.06337447464466095, 0.015428085811436176, 0.004312052857130766, -0.007229139097034931, 0.05527564883232117, -0.006808112375438213, 0.07088948786258698, -0.00012474505638238043, -0.04365086928009987, -0.04515659064054489, 0.038694124668836594, -0.013933437876403332, 0.017678912729024887, 0.012773140333592892, 0.006813810206949711, -0.07386386394500732, -0.037100885063409805, -0.04262838512659073, -0.007997148670256138, 0.0208552498370409, 0.04264616593718529, 0.017213813960552216, 0.014948179945349693, -0.07271534204483032, 0.0817916989326477, -0.03335314989089966, -0.03386693447828293, 0.009849249385297298, -0.050331465899944305, -0.0032846773974597454, 0.025923816487193108, -0.04490397870540619, -0.09390857815742493, -0.017283419147133827, 0.12618014216423035, 0.02509794570505619, 0.027086980640888214, -0.01584477722644806, 0.0073113166727125645, -0.04976154863834381, 0.06547246873378754, -0.0024508775677531958, 0.007991260848939419, 0.0266022440046072, -0.043695006519556046, 0.07472085952758789, 0.007441262248903513, 0.051124125719070435, 0.07752301543951035, -1.4638271927935875e-8, -0.0390424020588398, -0.008100203238427639, 0.009420936927199364, 0.036086998879909515, -0.0028061659540981054, 0.025474829599261284, -0.19067564606666565, 0.02547001838684082, -0.018264178186655045, -0.03138710930943489, 0.10895323008298874, -0.01719306968152523, -0.06447708606719971, -0.01600189507007599, 0.035164736211299896, 0.02779798023402691, -0.08427273482084274, 0.03713717684149742, -0.020861180499196053, -0.02191789075732231, -0.004857034422457218, 0.04417064040899277, 0.011752741411328316, 0.01457670982927084, -0.057050399482250214, -0.08057313412427902, 0.07217204570770264, 0.053681984543800354, -0.015566468238830566, -0.01801067404448986, 0.002736173802986741, 0.10317791253328323, -0.03227155655622482, 0.02699151448905468, -0.12257373332977295, 0.09017696976661682, -0.07745634019374847, 0.024419095367193222, -0.015739135444164276, -0.027195798233151436, 0.055806178599596024, -0.0700400248169899, -0.015378367155790329, -0.03542008996009827, -0.05105265974998474, -0.023523516952991486, 0.042941004037857056, -0.033636871725320816, 0.07079657167196274, -0.012599188834428787, -0.010406016372144222, 0.0241482462733984, -0.010708633810281754, 0.04295923560857773, -0.039465948939323425, -0.023630255833268166, -0.0626218318939209, -0.030449913814663887, 0.05210663005709648, 0.031663428992033005, -0.012112004682421684, -0.05480539798736572, 0.09829246997833252, -0.02818920463323593]) YIELD node, score RETURN node.`content` AS text, score, node } AS metadata